# Seq2Seq workshop — `mid` on WMT'14 En→Fr

Hands-on walkthrough of [Sutskever, Vinyals & Le, *Sequence to Sequence Learning with Neural Networks*](https://arxiv.org/abs/1409.3215) ([HTML](https://arxiv.org/html/1409.3215v3)), implemented in `src/seq2seq/`.

We run the **`mid`** preset: same *architecture and training recipe* as the paper, scaled to one GPU (~30–60 min).

### What the paper claims (§1–§2)

Map a variable-length source sequence $x=(x_1,\ldots,x_T)$ to a variable-length target $y=(y_1,\ldots,y_{T'})$ with a **single neural net**:

$$
p(y_1,\ldots,y_{T'}\mid x)=\prod_{t=1}^{T'} p(y_t\mid v,\,y_{<t}),
\quad v = q(x_1,\ldots,x_T).
$$

- Encoder $q$: deep LSTM reading $x$ → fixed vector $v$ (sentence embedding).
- Decoder: deep LSTM language model **conditioned on $v$** (initial state), emitting target tokens one by one.
- Softmax over the full target vocabulary at each step.

```
x (reversed) ──► Encoder LSTM×L ──► v = (h_T, c_T) ──► Decoder LSTM×L ──► softmax(|V_tgt|)
                                                        ▲
                                                 y_<t> (teacher / beam)
```

### Specs: paper experiment vs this workshop (`mid`)

| Knob | Paper (§2–§3.4) | **`mid` (this run)** |
|------|-----------------|----------------------|
| Task | WMT'14 En→Fr | same |
| Layers × hidden | 4 × 1000 | **4 × 256** |
| Embedding dim | 1000 | **256** |
| \|V_src\| / \|V_tgt\| | 160k / 80k | **20k / 20k** |
| Train pairs | full WMT (~12M) | **150k** (HF slice) |
| Max length | 100 | **50** |
| Batch | 128 | **64** |
| Epochs | 7.5 | **3** |
| LR | 0.7; hold 5 ep; ×½ / 0.5 ep | same **0.7**; hold **2** ep; ×½ / 0.5 ep |
| Grad clip | $\|g\|_2>5$ | same |
| Init | Unif$[-0.08,0.08]$ | same |
| Reverse source | yes | yes |
| Hardware | 8 GPUs, layer parallel | **1× CUDA GPU** |

**Epochs.** One epoch = one pass over the training pairs. The paper runs 7.5 passes so the model sees the full WMT corpus many times; `mid` stops at 3 because the subset is smaller and we are targeting a short single-GPU wall clock, not paper BLEU.

**Learning rate.** SGD starts at $0.7$ (paper §3.4). It stays flat through `lr_hold_epochs`, then halves every `lr_halve_every_epochs` (0.5). Paper holds for 5 epochs; `mid` holds for 2 so decay still kicks in inside a 3-epoch budget. That matches the *shape* of their schedule (flat → stepwise ×½), not their absolute duration.

**Grad clip.** If the global gradient norm $\|g\|_2$ exceeds 5, rescale $g \leftarrow 5\,g/\|g\|_2$. Deep LSTMs with large vocab softmaxes produce unstable updates; clipping is the paper's fixed safeguard (same threshold in `mid`).

Artifacts after train: `runs/mid/checkpoint.pt`, `runs/mid/monitor_history.json`.


## 0. Environment

CUDA required. On a Runpod official PyTorch image, **do not** reinstall torch:

```bash
cd /workspace/seq2seq
pip install -e ".[dev]" --no-deps
pip install datasets numpy tqdm pytest
```


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from seq2seq.config import mid_config, paper_config
from seq2seq.data import basic_tokenize, prepare_data, SPECIAL_TOKENS
from seq2seq.lstm_cell import LSTMCell
from seq2seq.deep_lstm import DeepLSTM
from seq2seq.model import Seq2Seq
from seq2seq.train import train, learning_rate_at_epoch, clip_grad_norm_
from seq2seq.decode import (
    load_checkpoint,
    greedy_decode,
    beam_search_decode,
    encode_source_sentence,
)

import torch

assert torch.cuda.is_available(), "Need CUDA for mid"
device = torch.device("cuda")
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

cfg = mid_config()
cfg.device = "cuda"
cfg.checkpoint_dir = str(ROOT / "runs" / "mid")
CKPT = Path(cfg.checkpoint_dir) / "checkpoint.pt"
HIST = Path(cfg.checkpoint_dir) / "monitor_history.json"
print("checkpoint:", CKPT, "exists=", CKPT.exists())


## 1. Hyperparameters — locked `mid` knobs

Every number below is what the training run actually used (`mid_config()`). Compare to `paper_config()` for the published scale.


In [ ]:
mid, paper = mid_config(), paper_config()

def row(label, a, b):
    print(f"{label:22} paper={a!s:>12}   mid={b!s:>12}")

row("num_layers", paper.num_layers, mid.num_layers)
row("hidden_size", paper.hidden_size, mid.hidden_size)
row("embed_size", paper.embed_size, mid.embed_size)
row("src_vocab_size", paper.src_vocab_size, mid.src_vocab_size)
row("tgt_vocab_size", paper.tgt_vocab_size, mid.tgt_vocab_size)
row("max_src/tgt_len", f"{paper.max_src_len}/{paper.max_tgt_len}", f"{mid.max_src_len}/{mid.max_tgt_len}")
row("max_train_examples", paper.max_train_examples, mid.max_train_examples)
row("batch_size", paper.batch_size, mid.batch_size)
row("epochs", paper.epochs, mid.epochs)
row("learning_rate", paper.learning_rate, mid.learning_rate)
row("lr_hold_epochs", paper.lr_hold_epochs, mid.lr_hold_epochs)
row("lr_halve_every", paper.lr_halve_every_epochs, mid.lr_halve_every_epochs)
row("grad_clip", paper.grad_clip, mid.grad_clip)
row("init_range", paper.init_range, mid.init_range)
row("reverse_source", paper.reverse_source, mid.reverse_source)
row("beam_size", paper.beam_size, mid.beam_size)

print("\nLR schedule under mid (hold 2 ep, then halve every 0.5 ep):")
for ep in [0.0, 1.0, 2.0, 2.5, 3.0]:
    print(f"  epoch {ep:.1f} → LR = {learning_rate_at_epoch(mid, ep):.4g}")


## 2. Data pipeline (§3.1, §3.3, §3.4)

### Steps (code: `seq2seq.data.prepare_data`)

1. **Load** Hugging Face `wmt/wmt14`, config `fr-en` (English ↔ French). We map **En→Fr**.
2. Cap at `max_train_examples=150_000` pairs (paper: full corpus).
3. **Tokenize** with whitespace (`basic_tokenize`) — paper-era word tokens, not BPE.
4. **Filter** pairs with $1 \le |x| \le 50$, $1 \le |y| \le 50$.
5. **Vocabularies** frequency-capped at 20k each; specials `<PAD> <UNK> <EOS> <SOS>` reserved; OOV → `<UNK>`.
6. **Reverse source** word order (targets unchanged) — paper §2 / §3.3: shortens the minimum time lag between related tokens; large long-sentence BLEU effect.
7. Encode IDs + append `<EOS>`; decoder training prepends `<SOS>` in the collate.
8. **Length-bucketed** minibatches (`bucket_width=5`) — paper §3.4 ≈2× speedup vs random padding waste.

```
EN:  the cat sat on the mat
ENC: mat the on sat cat the  + <EOS>  →  encoder
DEC: <SOS> le chat … <EOS>           →  teacher-forced targets
```

Set `PEEK_DATA = True` to download and inspect (slow first time). If you already have a checkpoint, leave it `False` — vocabs are stored in the checkpoint.


In [ ]:
PEEK_DATA = False  # True → download WMT slice + print one batch

if PEEK_DATA:
    loader, src_vocab, tgt_vocab, examples = prepare_data(cfg)
    print(f"pairs after filter: {len(examples):,}")
    print(f"|V_src|={len(src_vocab)}  |V_tgt|={len(tgt_vocab)}  specials={SPECIAL_TOKENS}")
    ex = examples[0]
    print(f"example 0: src_len={ex.src_len} tgt_len={ex.tgt_len}")
    # Show reversal on a raw English string (same rule as encode_source_sentence)
    raw = "the cat sat on the mat"
    toks = basic_tokenize(raw)
    print("tokens:  ", toks)
    print("encoder: ", list(reversed(toks)))
    batch = next(iter(loader))
    for k, v in batch.items():
        print(f"  batch[{k!r}]: {tuple(v.shape)}")
else:
    print("Skipped HF download. Will use vocabs from checkpoint in §6.")


## 3. Graves LSTM cell (§2; Graves 2013)

Each layer is an LSTM with input / forget / candidate / output gates. Implementation: `lstm_cell.py` (gates packed as $[i\,|\,f\,|\,g\,|\,o]$).

$$
\begin{aligned}
i_t &= \sigma(W_{xi}x_t + W_{hi}h_{t-1} + b_i) \\
f_t &= \sigma(W_{xf}x_t + W_{hf}h_{t-1} + b_f) \\
g_t &= \tanh(W_{xg}x_t + W_{hg}h_{t-1} + b_g) \\
o_t &= \sigma(W_{xo}x_t + W_{ho}h_{t-1} + b_o) \\
c_t &= f_t \odot c_{t-1} + i_t \odot g_t \\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

At **`mid`**: each cell has hidden size $H=256$. Layer 0 input dim = embed $E=256$; layers $1..3$ input dim $=H$.


In [ ]:
# One cell step at mid width
cell = LSTMCell(input_size=cfg.embed_size, hidden_size=cfg.hidden_size).to(device)
B = 2
x = torch.randn(B, cfg.embed_size, device=device)
h0 = torch.zeros(B, cfg.hidden_size, device=device)
c0 = torch.zeros(B, cfg.hidden_size, device=device)
h1, c1 = cell(x, (h0, c0))
print("x", tuple(x.shape), "→ h,c", tuple(h1.shape), tuple(c1.shape))
n_params = sum(p.numel() for p in cell.parameters())
print(f"params per cell (mid): {n_params:,}")


## 4. Deep LSTM stack (`deep_lstm.py`)

Paper: **4 layers**. For each timestep $t$:

$$
\begin{aligned}
u^{(0)}_t &= e_t && \text{(embedding)} \\
(h^{(\ell)}_t, c^{(\ell)}_t) &= \mathrm{LSTM}^{(\ell)}(u^{(\ell)}_t,\, h^{(\ell)}_{t-1},\, c^{(\ell)}_{t-1}) \\
u^{(\ell+1)}_t &= h^{(\ell)}_t
\end{aligned}
$$

for $\ell=0,\ldots,L-1$. Length masking zeros updates past true sequence end.

Encoder output used as sentence vector: final states $(h_T^{(\ell)}, c_T^{(\ell)})$ for **all** layers $\ell$ — we copy the full stack into the decoder (repo convention; paper emphasises the top-layer $v$).


In [ ]:
stack = DeepLSTM(cfg.embed_size, cfg.hidden_size, cfg.num_layers).to(device)
T, B = 7, 3
x = torch.randn(B, T, cfg.embed_size, device=device)
lengths = torch.tensor([7, 5, 3], device=device)
out, (h, c) = stack(x, lengths=lengths)
print("out (B,T,H):", tuple(out.shape))
print("h,c (L,B,H):", tuple(h.shape), tuple(c.shape))
print(f"DeepLSTM params: {sum(p.numel() for p in stack.parameters()):,}")


## 5. Full Seq2Seq model (`model.py`)

Separate encoder / decoder deep LSTMs (paper §2 — not weight-tied).

| Module | Spec (`mid`) |
|--------|----------------|
| `src_embed` | $\|V_{\mathrm{src}}\| \times 256$ |
| `encoder` | DeepLSTM $L=4$, $H=256$ |
| `tgt_embed` | $\|V_{\mathrm{tgt}}\| \times 256$ |
| `decoder` | DeepLSTM $L=4$, $H=256$ |
| `out_proj` | $256 \to \|V_{\mathrm{tgt}}\|$ (naive softmax) |

**Training forward (teacher forcing):**

1. Embed + encode source → state $(h,c)$.
2. For each target step $t$: embed gold $y_{t-1}$ (with SOS at $t=1$), one decoder step, project to logits.
3. Loss = mean NLL over non-pad target tokens:

$$
\mathcal{L} = -\frac{1}{N}\sum_{n,t} \mathbf{1}_{y_{n,t}\neq\mathrm{PAD}}\log p(y_{n,t}\mid v_n, y_{n,<t}).
$$

**Init:** $\mathrm{Unif}[-0.08,0.08]$ on all parameters (paper §3.4).


In [ ]:
# Shapes with placeholder vocab sizes (real sizes come from data / checkpoint)
Vs, Vt = cfg.src_vocab_size, cfg.tgt_vocab_size
model = Seq2Seq.from_config(cfg, Vs, Vt, pad_id=0).to(device)
model.init_weights(cfg.init_range)
n = sum(p.numel() for p in model.parameters())
print(f"Seq2Seq params (V={Vs}/{Vt}): {n:,}  (~{n/1e6:.1f}M)")
print("groups:", list(model.param_groups_for_monitor()))

# Fake batch at mid dims
B, Ts, Tt = 4, 12, 10
src = torch.randint(1, Vs, (B, Ts), device=device)
tgt_in = torch.randint(1, Vt, (B, Tt), device=device)
tgt_out = torch.randint(1, Vt, (B, Tt), device=device)
src_lengths = torch.full((B,), Ts, device=device)
loss, logits = model(src, src_lengths, tgt_in, tgt_out)
print(f"loss={float(loss):.4f}  logits={tuple(logits.shape)}  (= B, T_tgt, V_tgt)")


## 6. Training recipe (§3.4) — what the loop does

Code: `seq2seq.train.train`.

| Step | Detail |
|------|--------|
| Optimiser | SGD, **no momentum**, LR from schedule |
| Gradients | already batch-mean from NLL; clip if $\|g\|_2 > 5$: $g \leftarrow 5\,g/\|g\|_2$ |
| LR | $0.7$ until epoch 2; then halve every 0.5 epoch |
| Logging | every 20 steps: loss, ppl$=e^{\mathrm{loss}}$, $\|g\|$, $\|\Delta\theta\|$ by group; sample decode every 100 |
| Checkpoint | `runs/mid/checkpoint.pt` (weights + vocabs + config) + `monitor_history.json` |

If you already finished CLI train, leave `DO_TRAIN = False` and load the checkpoint.


In [ ]:
DO_TRAIN = False  # True → ~30–60 min full mid train

if DO_TRAIN:
    monitor = train(cfg, synthetic=False, sample_src="the cat sat on the mat")
    ckpt = load_checkpoint(CKPT, device="cuda")
else:
    if not CKPT.exists():
        raise FileNotFoundError(
            f"Missing {CKPT}. Train first:\n"
            "  python -m seq2seq.train --config mid --device cuda"
        )
    ckpt = load_checkpoint(CKPT, device="cuda")

src_vocab = ckpt["src_vocab"]
tgt_vocab = ckpt["tgt_vocab"]
# Rebuild with *actual* vocab sizes from the checkpoint
model = Seq2Seq.from_config(cfg, len(src_vocab), len(tgt_vocab), tgt_vocab.pad_id)
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
print(
    f"loaded step={ckpt.get('step')}  "
    f"|V_src|={len(src_vocab)} |V_tgt|={len(tgt_vocab)}  "
    f"params={sum(p.numel() for p in model.parameters()):,}"
)


## 7. Training dynamics

`monitor_history.json` records the §3.4 loop: NLL, perplexity, LR, grad norms, parameter-group updates. Under `mid` scale, perplexity stays high and samples stay rough — we check that **loss falls** and the pipeline is paper-faithful, not that we hit 34.8 BLEU.


In [ ]:
if not HIST.exists():
    print("No monitor_history.json — was train interrupted before save?")
else:
    history = json.loads(HIST.read_text())
    print(f"{len(history)} log rows")
    last = history[-1]
    print(
        f"final: step={last['step']} epoch={last['epoch']:.3f} "
        f"loss={last['loss']:.4f} ppl={last['ppl']:.2f} lr={last['lr']:.4g}"
    )
    if last.get("sample"):
        print("last sample:", last["sample"])


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("install matplotlib to plot")

if plt and HIST.exists():
    history = json.loads(HIST.read_text())
    steps = [r["step"] for r in history]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    axes[0].plot(steps, [r["loss"] for r in history], lw=1)
    axes[0].set(title="token NLL", xlabel="step", ylabel="loss")
    axes[1].plot(steps, [r["ppl"] for r in history], lw=1)
    axes[1].set(title="perplexity", xlabel="step", ylabel="ppl")
    axes[1].set_yscale("log")
    axes[2].plot(steps, [r["lr"] for r in history], lw=1)
    axes[2].set(title="learning rate", xlabel="step", ylabel="lr")
    fig.suptitle("mid training (§3.4 schedule)", y=1.02)
    plt.tight_layout()
    plt.show()


## 8. Decoding (§3.2)

At inference the decoder is **not** teacher-forced:

1. Encode reversed source → $v$.
2. Start from `<SOS>`; at each step take $\arg\max$ (**greedy**) or keep a beam of partial hypotheses (**beam search**).
3. Stop at `<EOS>` or `max_decode_len`.

Paper: left-to-right beam; **beam size 2** captures most of the gain over greedy. Softmax is still over the full target vocab (no class-factored / sampled softmax in this replica).

Expect imperfect French / `<UNK>` / repetition at `mid` capacity — that is the scale limit, not a broken decode.


In [ ]:
sentences = [
    "the cat sat on the mat",
    "hello world",
    "I love neural networks",
    "this is a longer sentence about machine translation",
]

for text in sentences:
    src_t, lens = encode_source_sentence(
        text, src_vocab, reverse=cfg.reverse_source, device=device
    )
    # Show what the encoder actually sees
    toks = basic_tokenize(text)
    enc_order = list(reversed(toks))
    g = greedy_decode(model, src_t, lens, tgt_vocab.sos_id, tgt_vocab.eos_id, cfg.max_decode_len)[0]
    b = beam_search_decode(
        model, src_t, lens, tgt_vocab.sos_id, tgt_vocab.eos_id,
        cfg.max_decode_len, beam_size=cfg.beam_size,
    )[0]
    print(f"EN:      {text}")
    print(f"ENC in:  {' '.join(enc_order)}")
    print(f"greedy:  {tgt_vocab.decode(g)}")
    print(f"beam-{cfg.beam_size}:  {tgt_vocab.decode(b)}")
    print()


## 9. Mapping back to the paper

| Paper section | What we implemented | What `mid` changes |
|---------------|---------------------|--------------------|
| §2 Objective | $p(y\mid x)=\prod_t p(y_t\mid v,y_{<t})$ | unchanged |
| §2 Architecture | 4-layer LSTM enc/dec, separate | $H{=}1000\to256$ |
| §2 Reversal | source reversed | unchanged |
| §3.1 Data | WMT En→Fr, word vocab | 150k pairs, 20k/20k, len≤50 |
| §3.2 Decode | beam search | beam 2 |
| §3.4 Train | SGD 0.7, clip 5, Unif init, buckets | hold 2 ep, 3 ep total |
| §3–§4 Scale | ~384M params, 8-GPU, ensemble | ~single-digit–tens of M params, 1 GPU |
| Results | 34.8 BLEU (ensemble) | pedagogical — not a BLEU claim |

**Out of scope for this replica:** 8-GPU layer / softmax parallel; 5-model ensemble; SMT 1000-best rescoring; claiming paper BLEU.

### References

1. Sutskever, Vinyals, Le. *Sequence to Sequence Learning with Neural Networks*. NeurIPS 2014. [arXiv:1409.3215](https://arxiv.org/abs/1409.3215)
2. Graves. *Generating Sequences With Recurrent Neural Networks*. 2013. [arXiv:1308.0850](https://arxiv.org/abs/1308.0850)
